In [6]:
real_world_urls = [

    # ===============================
    # 1. Popular Legitimate Websites
    # ===============================

    ("https://www.google.com", 1, "Popular"),
    ("https://www.youtube.com", 1, "Popular"),
    ("https://www.wikipedia.org", 1, "Popular"),
    ("https://www.amazon.com", 1, "Popular"),
    ("https://www.apple.com", 1, "Popular"),
    ("https://www.microsoft.com", 1, "Popular"),
    ("https://www.adobe.com", 1, "Popular"),
    ("https://www.ibm.com", 1, "Popular"),
    ("https://www.netflix.com", 1, "Popular"),
    ("https://www.spotify.com", 1, "Popular"),


    # ===============================
    # 2. Developer / Documentation
    # Original False Positive Area
    # ===============================

    ("https://developer.mozilla.org/en-US/docs/Web/API", 1, "Documentation"),
    ("https://numpy.org/doc/stable/reference", 1, "Documentation"),
    ("https://pytorch.org/tutorials", 1, "Documentation"),
    ("https://docs.python.org/3/", 1, "Documentation"),
    ("https://scikit-learn.org/stable/user_guide.html", 1, "Documentation"),
    ("https://kubernetes.io/docs/home/", 1, "Documentation"),
    ("https://flask.palletsprojects.com/en/stable/", 1, "Documentation"),
    ("https://docs.docker.com/", 1, "Documentation"),
    ("https://docs.aws.amazon.com/", 1, "Documentation"),
    ("https://react.dev/learn", 1, "Documentation"),
    ("https://nodejs.org/docs/latest/api/", 1, "Documentation"),
    ("https://www.tensorflow.org/tutorials", 1, "Documentation"),


    # ===============================
    # 3. Open Source / Developer Platforms
    # ===============================

    ("https://github.com/openai/openai-python", 1, "Open Source"),
    ("https://github.com/scikit-learn/scikit-learn", 1, "Open Source"),
    ("https://github.com/pallets/flask", 1, "Open Source"),
    ("https://github.com/tensorflow/tensorflow", 1, "Open Source"),
    ("https://gitlab.com/gitlab-org/gitlab", 1, "Open Source"),
    ("https://pypi.org/project/requests/", 1, "Open Source"),
    ("https://www.npmjs.com/package/react", 1, "Open Source"),


    # ===============================
    # 4. Education / Government
    # ===============================

    ("https://www.harvard.edu", 1, "Education"),
    ("https://www.mit.edu", 1, "Education"),
    ("https://www.ox.ac.uk", 1, "Education"),
    ("https://www.cam.ac.uk", 1, "Education"),

    ("https://www.gov.uk", 1, "Government"),
    ("https://www.nasa.gov", 1, "Government"),
    ("https://www.usa.gov", 1, "Government"),


    # ===============================
    # 5. Finance / Services
    # ===============================

    ("https://www.paypal.com", 1, "Finance"),
    ("https://www.bankofamerica.com", 1, "Finance"),
    ("https://www.chase.com", 1, "Finance"),
    ("https://www.visa.com", 1, "Finance"),


    # ===============================
    # 6. Login / Account Pages
    # Legitimate but suspicious-looking
    # ===============================

    ("https://accounts.google.com", 1, "Login"),
    ("https://login.microsoftonline.com", 1, "Login"),
    ("https://github.com/login", 1, "Login"),
    ("https://www.amazon.com/ap/signin", 1, "Login"),


    # ===============================
    # 7. Known / Simulated Phishing
    # Expected: Phishing = 0
    # ===============================

    ("http://paypal-login-security-check.test", 0, "Phishing"),
    ("http://microsoft-account-verification.test", 0, "Phishing"),
    ("http://paypa1-secure-login.win", 0, "Phishing"),
    ("http://google-security-update-login.com", 0, "Phishing"),
    ("http://amazon-account-verify.net", 0, "Phishing"),
    ("http://apple-id-confirmation.com", 0, "Phishing"),
    ("http://secure-bank-login-alert.com", 0, "Phishing"),
    ("http://facebook-security-check.com", 0, "Phishing"),
]

In [7]:
import pandas as pd

test_df = pd.DataFrame(
    real_world_urls,
    columns=[
        "url",
        "actual",
        "category"
    ]
)

test_df.head()

,url,actual,category
0,https://www.google.com,1,Popular
1,https://www.youtube.com,1,Popular
2,https://www.wikipedia.org,1,Popular
3,https://www.amazon.com,1,Popular
4,https://www.apple.com,1,Popular


In [8]:
import sys
import os

sys.path.append("../backend")


In [9]:
from app.features import (
    compute_features,
    URL_ONLY_FEATURES
)

In [10]:
print(URL_ONLY_FEATURES)

['URLLength', 'DomainLength', 'IsDomainIP', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS']


In [11]:
import joblib

tld_lookup = joblib.load(
    "../outputs/tld_probability_lookup.joblib"
)

char_lookup = joblib.load(
    "../outputs/char_probability_lookup.joblib"
)

In [12]:
rf_new = joblib.load(
    "../outputs/rf_model_v2_dataset_improved.joblib"
)

In [13]:
print(type(rf_new))
print(len(URL_ONLY_FEATURES))
print(len(tld_lookup))
print(len(char_lookup))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
20
709
100


In [14]:
import numpy as np


results = []


for _, row in test_df.iterrows():

    url = row["url"]

    features = compute_features(
        url,
        tld_lookup,
        char_lookup
    )

    vector = np.array(
        [
            [
                features[name]
                for name in URL_ONLY_FEATURES
            ]
        ]
    )


    prediction = rf_new.predict(vector)[0]

    probability = rf_new.predict_proba(vector)[0]


    results.append({

        "url": url,

        "category": row["category"],

        "actual": row["actual"],

        "prediction": prediction,

        "phishing_probability": round(
            probability[0],
            3
        ),

        "legitimate_probability": round(
            probability[1],
            3
        ),

        "correct":
            prediction == row["actual"]

    })


results_df = pd.DataFrame(results)

results_df

c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\De

,url,category,actual,prediction,phishing_probability,legitimate_probability,correct
0,https://www.google.com,Popular,1,1,0.00,1.00,True
1,https://www.youtube.com,Popular,1,1,0.00,1.00,True
2,https://www.wikipedia.org,Popular,1,1,0.00,1.00,True
3,https://www.amazon.com,Popular,1,1,0.00,1.00,True
4,https://www.apple.com,Popular,1,1,0.00,1.00,True
5,https://www.microsoft.com,Popular,1,1,0.00,1.00,True
6,https://www.adobe.com,Popular,1,1,0.00,1.00,True
7,https://www.ibm.com,Popular,1,1,0.00,1.00,True
8,https://www.netflix.com,Popular,1,1,0.00,1.00,True
9,https://www.spotify.com,Popular,1,1,0.00,1.00,True


In [15]:
accuracy = (
    results_df["correct"].mean()
)

print(
    "Overall Accuracy:",
    round(accuracy*100,2),
    "%"
)

Overall Accuracy: 75.0 %


In [16]:
category_result = (
    results_df
    .groupby("category")
    ["correct"]
    .mean()
    *100
)

category_result

category
Documentation     41.666667
Education        100.000000
Finance          100.000000
Government       100.000000
Login             50.000000
Open Source       42.857143
Phishing         100.000000
Popular          100.000000
Name: correct, dtype: float64

In [17]:
results_df[
    results_df["correct"] == False
]

,url,category,actual,prediction,phishing_probability,legitimate_probability,correct
10,https://developer.mozilla.org/en-US/docs/Web/API,Documentation,1,0,0.89,0.11,False
16,https://flask.palletsprojects.com/en/stable/,Documentation,1,0,1.00,0.00,False
17,https://docs.docker.com/,Documentation,1,0,0.90,0.10,False
18,https://docs.aws.amazon.com/,Documentation,1,0,0.50,0.50,False
19,https://react.dev/learn,Documentation,1,0,0.89,0.11,False
20,https://nodejs.org/docs/latest/api/,Documentation,1,0,0.79,0.21,False
21,https://www.tensorflow.org/tutorials,Documentation,1,0,0.77,0.23,False
24,https://github.com/pallets/flask,Open Source,1,0,0.99,0.01,False
25,https://github.com/tensorflow/tensorflow,Open Source,1,0,1.00,0.00,False
26,https://gitlab.com/gitlab-org/gitlab,Open Source,1,0,1.00,0.00,False


In [18]:
failed_urls = [
    "https://developer.mozilla.org/en-US/docs/Web/API",
    "https://flask.palletsprojects.com/en/stable/",
    "https://docs.docker.com/",
    "https://github.com/pallets/flask",
    "https://github.com/login",
    "https://www.amazon.com/ap/signin"
]


for url in failed_urls:
    print("\nURL:", url)

    print(
        compute_features(
            url,
            tld_lookup,
            char_lookup
        )
    )


URL: https://developer.mozilla.org/en-US/docs/Web/API
{'URLLength': 48, 'DomainLength': 21, 'IsDomainIP': 0, 'TLDLegitimateProb': 0.8790764483694207, 'URLCharProb': 0.047091950469852074, 'TLDLength': 3, 'NoOfSubDomain': 1, 'HasObfuscation': 0, 'NoOfObfuscatedChar': 0, 'ObfuscationRatio': 0.0, 'NoOfLettersInURL': 38, 'LetterRatioInURL': 0.792, 'NoOfDegitsInURL': 0, 'DegitRatioInURL': 0.0, 'NoOfEqualsInURL': 0, 'NoOfQMarkInURL': 0, 'NoOfAmpersandInURL': 0, 'NoOfOtherSpecialCharsInURL': 1, 'SpacialCharRatioInURL': 0.021, 'IsHTTPS': 1}

URL: https://flask.palletsprojects.com/en/stable/
{'URLLength': 44, 'DomainLength': 25, 'IsDomainIP': 0, 'TLDLegitimateProb': 0.6104888525986936, 'URLCharProb': 0.05212533487198335, 'TLDLength': 3, 'NoOfSubDomain': 1, 'HasObfuscation': 0, 'NoOfObfuscatedChar': 0, 'ObfuscationRatio': 0.0, 'NoOfLettersInURL': 36, 'LetterRatioInURL': 0.818, 'NoOfDegitsInURL': 0, 'DegitRatioInURL': 0.0, 'NoOfEqualsInURL': 0, 'NoOfQMarkInURL': 0, 'NoOfAmpersandInURL': 0, 'NoOfO

In [20]:
from app.features import (
    compute_features,
    feature_vector,
    URL_ONLY_FEATURES
)


In [21]:
rf_new.predict_proba(
    feature_vector("https://developer.mozilla.org/en-US/docs/Web/API")
)

c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


array([[0.89, 0.11]])

In [22]:
failed_urls = [
    "https://flask.palletsprojects.com/en/stable/",
    "https://docs.docker.com/",
    "https://github.com/pallets/flask",
    "https://github.com/login",
    "https://www.amazon.com/ap/signin"
]


for url in failed_urls:
    print("\nURL:", url)

    print(
        rf_new.predict_proba(
            feature_vector(url)
        )
    )


URL: https://flask.palletsprojects.com/en/stable/
[[1. 0.]]

URL: https://docs.docker.com/
[[0.9 0.1]]

URL: https://github.com/pallets/flask
[[0.99 0.01]]

URL: https://github.com/login
[[0.99 0.01]]

URL: https://www.amazon.com/ap/signin
[[0.95 0.05]]


c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Inti course folder\De

In [23]:
from sklearn.metrics import confusion_matrix, classification_report


y_true = results_df["actual"]
y_pred = results_df["prediction"]


cm = confusion_matrix(
    y_true,
    y_pred
)


print("Confusion Matrix")
print(cm)


print("\nClassification Report")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Phishing",
            "Legitimate"
        ]
    )
)

Confusion Matrix
[[ 8  0]
 [13 31]]

Classification Report
              precision    recall  f1-score   support

    Phishing       0.38      1.00      0.55         8
  Legitimate       1.00      0.70      0.83        44

    accuracy                           0.75        52
   macro avg       0.69      0.85      0.69        52
weighted avg       0.90      0.75      0.78        52



In [24]:
from sklearn.metrics import confusion_matrix


cm = confusion_matrix(
    y_true,
    y_pred
)


TN = cm[0][0]
FP = cm[1][0]
FN = cm[0][1]
TP = cm[1][1]


false_positive_rate = FP / (FP + TN)

false_negative_rate = FN / (FN + TP)


print("False Positive Count:", FP)

print(
    "False Positive Rate:",
    round(false_positive_rate*100,2),
    "%"
)


print("False Negative Count:", FN)

print(
    "False Negative Rate:",
    round(false_negative_rate*100,2),
    "%"
)

False Positive Count: 13
False Positive Rate: 61.9 %
False Negative Count: 0
False Negative Rate: 0.0 %


In [25]:
category_summary = (
    results_df
    .groupby("category")
    .agg(
        total=("correct","count"),
        correct=("correct","sum")
    )
)


category_summary["accuracy"] = (
    category_summary["correct"]
    /
    category_summary["total"]
    *
    100
)


category_summary

,total,correct,accuracy
category,,,
Documentation,12,5,41.666667
Education,4,4,100.000000
Finance,4,4,100.000000
Government,3,3,100.000000
Login,4,2,50.000000
Open Source,7,3,42.857143
Phishing,8,8,100.000000
Popular,10,10,100.000000


In [26]:
false_positive_cases = results_df[
    (results_df["actual"] == 1) &
    (results_df["prediction"] == 0)
]


fp_summary = (
    false_positive_cases
    .groupby("category")
    .size()
)


fp_summary

category
Documentation    7
Login            2
Open Source      4
dtype: int64